In [12]:
# 07_tokenization.ipynb (complete)
import os
import pandas as pd
from transformers import AutoTokenizer

# make sure cwd is repo root (optional)
# os.chdir(r"D:\Final Year Project\somali-nlp-research")

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
MAX_LEN = 256

train = pd.read_csv("../data/processed/clean_train.csv")
val   = pd.read_csv("../data/processed/clean_val.csv")
test  = pd.read_csv("../data/processed/clean_test.csv")

def tok(texts: pd.Series):
    return tokenizer(
        texts.astype(str).tolist(),
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

train_enc = tok(train["Text"])
val_enc   = tok(val["Text"])
test_enc  = tok(test["Text"])

# labels -> ids
label2id = {"HUMAN": 0, "AI": 1}
y_train = train["Label"].astype(str).str.strip().str.upper().map(label2id).tolist()
y_val   = val["Label"].astype(str).str.strip().str.upper().map(label2id).tolist()
y_test  = test["Label"].astype(str).str.strip().str.upper().map(label2id).tolist()

# final tokenized tables
train_tok = pd.DataFrame({
    "input_ids": train_enc["input_ids"],
    "attention_mask": train_enc["attention_mask"],
    "label": y_train,
})
val_tok = pd.DataFrame({
    "input_ids": val_enc["input_ids"],
    "attention_mask": val_enc["attention_mask"],
    "label": y_val,
})
test_tok = pd.DataFrame({
    "input_ids": test_enc["input_ids"],
    "attention_mask": test_enc["attention_mask"],
    "label": y_test,
})

# sanity checks
print("train/val/test sizes:", len(train_tok), len(val_tok), len(test_tok))
print("seq len:", len(train_tok.loc[0, "input_ids"]), len(train_tok.loc[0, "attention_mask"]))
print("label counts (train):", train_tok["label"].value_counts().to_dict())

# save for later use (fast reload)
train_tok.to_pickle("../data/processed/train_tok.pkl")
val_tok.to_pickle("../data/processed/val_tok.pkl")
test_tok.to_pickle("../data/processed/test_tok.pkl")

print("saved: ../data/processed/*_tok.pkl")

train/val/test sizes: 4127 884 884
seq len: 256 256
label counts (train): {1: 2092, 0: 2035}
saved: ../data/processed/*_tok.pkl
